# IPCA Monte Carlo simulations

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


# **IPCA NON-LINEAR total**

**Iteration 2**

In [ ]:
!pip install tables
!pip install ipca

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from ipca import InstrumentedPCA
from pathlib import Path
from tqdm import tqdm
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIMULATIONS_NON_LINEAR_TOTAL')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    print(f"Creating directory: {results_path}")
    results_path.mkdir(parents=True)
else:
    print(f"Directory already exists: {results_path}")

# Check if the directory was created successfully
if results_path.exists():
    print(f"Directory successfully created: {results_path}")
else:
    print(f"Failed to create directory: {results_path}")

# Parameters
N = 200
T = 180
Pc = 50
Px = 50
num_factors = 5
R2_target = 0.45
num_simulations = 100
factors = [1, 2, 3, 4, 5, 6]

cols = ['Factors', 'Sharpe Ratio x_t', 'Long-Short Portfolio Sharpe Ratio', 'R^2 for r_t', 'Adjusted R^2', 'R^2 for x_t']

# Function to generate data
def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([
            np.power(1.2 * c_normalized[:, 0, t], 3),
            c_normalized[:, 1, t] * c_normalized[:, 0, t],
            np.power(0.4 * c_normalized[:, 2, t], 2),
            2 * c_normalized[:, 3, t] * c_normalized[:, 2, t],
            np.power(0.6 * c_normalized[:, 4, t], 4)
        ]).T
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)
    test_size = T - train_size - val_size - 1
    train_data = r_adjusted[:, :train_size]
    val_data = r_adjusted[:, train_size:train_size + val_size]
    test_data = r_adjusted[:, train_size + val_size:]
    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size

# Helper functions
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[2]):
        Z_t = Z_t_minus_1[:, :, t]
        r_t_i = r_t[:, t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

def calculate_sharpe_ratio(returns, risk_free_rate=0):
    mean_return = np.mean(returns)
    std_dev_return = np.std(returns)
    sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
    return sharpe_ratio

def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    N = predicted_returns.shape[0]
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold
    long_returns = actual_returns[long_positions].mean()
    short_returns = actual_returns[short_positions].mean()
    portfolio_return = long_returns - short_returns
    return portfolio_return

# Loop over each factor and calculate average R^2 for all simulations
scores_dict = {}
for n_factors in tqdm(factors):
    r2_scores = []
    scores = []
    for i in tqdm(range(num_simulations)):
        train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)
        X_train = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(1, train_size + 1)], axis=0)
        y_train = np.concatenate([train_data[:, t] for t in range(train_size)], axis=0)
        X_test = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(train_size + val_size + 1, T)], axis=0)
        y_test = np.concatenate([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)], axis=0)
        indices_train = np.array([[i, t] for t in range(1, train_size + 1) for i in range(N)])
        indices_test = np.array([[i, t] for t in range(train_size + val_size + 1, T) for i in range(N)])
        X_train_df = pd.DataFrame(X_train, index=pd.MultiIndex.from_arrays([indices_train[:, 0], indices_train[:, 1]], names=['entity', 'time']))
        y_train_series = pd.Series(y_train, index=pd.MultiIndex.from_arrays([indices_train[:, 0], indices_train[:, 1]], names=['entity', 'time']))
        X_test_df = pd.DataFrame(X_test, index=pd.MultiIndex.from_arrays([indices_test[:, 0], indices_test[:, 1]], names=['entity', 'time']))
        y_test_series = pd.Series(y_test, index=pd.MultiIndex.from_arrays([indices_test[:, 0], indices_test[:, 1]], names=['entity', 'time']))
        regr = InstrumentedPCA(n_factors=n_factors, intercept=False, max_iter=2000, iter_tol=1e-05, alpha=0.0, l1_ratio=1.0, n_jobs=-1, backend='loky')
        regr.fit(X=X_train_df, y=y_train_series)

        try:
            y_pred = regr.predict(X=X_test_df, mean_factor=True)
            y_pred = np.array(y_pred).flatten()
            r2 = r2_score(y_test_series, y_pred)
            r2_scores.append(r2)
            Z_t_minus_1_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_size + 1, T)])
            r_t_test = np.array([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)])
            factor_network_input_test = calculate_factor_network_input(Z_t_minus_1_test, r_t_test)
            sharpe_ratio_x_t = calculate_sharpe_ratio(factor_network_input_test)
            y_pred_reshaped = y_pred.reshape((r_t_test.shape[0], r_t_test.shape[1]))
            factor_network_input_pred = calculate_factor_network_input(Z_t_minus_1_test, y_pred_reshaped)
            r2_x_t = r2_score(factor_network_input_test, factor_network_input_pred)
            portfolio_returns = []
            for t in range(r_t_test.shape[1]):
                actual_returns = r_t_test[:, t]
                predicted_returns = y_pred_reshaped[:, t]
                if predicted_returns.shape == actual_returns.shape:
                    portfolio_return = calculate_long_short_portfolio(predicted_returns, actual_returns)
                    portfolio_returns.append(portfolio_return)
                else:
                    print(f"Shape mismatch: predicted_returns.shape = {predicted_returns.shape}, actual_returns.shape = {actual_returns.shape}")
            portfolio_returns = np.array(portfolio_returns)
            long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)
            prediction_df = pd.DataFrame({
                'predicted returns': y_pred,
                'actual returns': y_test_series
            })
            print(f'| FACTOR : {n_factors} | SIMULATION: {i} | TEST R^2, r_t: {r2} |TEST R^2, x_t : {r2_x_t} ')
            print(f"Simulation {i+1}, Factors {n_factors} - R-squared (R²): {r2}")
            scores.append([n_factors, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, r2, adjusted_R2, r2_x_t])
        except ValueError as e:
            print(f"Simulation {i+1}, Factors {n_factors} - Error during prediction: {e}")

    # Save scores to HDF5
    scores_df = pd.DataFrame(scores, columns=cols)
    scores_dict[n_factors] = scores_df
    hdf5_path = results_path / f'scores_{n_factors}.h5'
    print(f"Saving scores to {hdf5_path}")
    try:
        with pd.HDFStore(hdf5_path, mode='w') as store:
            store.put('scores', scores_df)
            store.get_storer('scores').attrs.metadata = {'n_factors': n_factors}
        print(f"Scores successfully saved to {hdf5_path}")
    except Exception as e:
        print(f"Error saving scores to {hdf5_path}: {e}")



**load results**

In [ ]:
import pandas as pd
from pathlib import Path
!pip install tables
!pip install ipca
from google.colab import drive
drive.mount('/content/drive')



# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIMULATIONS_NON_LINEAR_PREDICTIVE')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# List of factors used in the simulations
factors = [1, 3, 4, 5, 6]

# Dictionary to store the loaded scores
loaded_scores_dict = {}

# Load the scores for each factor
for n_factors in factors:
    hdf5_path = results_path / f'scores_{n_factors}.h5'
    try:
        # Load the scores from the HDF5 file
        scores_df = pd.read_hdf(str(hdf5_path), key='scores')
        loaded_scores_dict[n_factors] = scores_df
        print(f"Scores successfully loaded for {n_factors} factors:")
        print(scores_df)
    except Exception as e:
        print(f"Error loading scores from {hdf5_path}: {e}")

# Access the loaded scores for further analysis
# Example: Access scores for a specific factor
specific_factor = 3
if specific_factor in loaded_scores_dict:
    specific_scores_df = loaded_scores_dict[specific_factor]
    # Perform operations on specific_scores_df
    print(f"Scores for factor {specific_factor}:")
    print(specific_scores_df)
else:
    print(f"No scores found for factor {specific_factor}")


In [ ]:
import pandas as pd
from pathlib import Path
!pip install tables
!pip install ipca
from google.colab import drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIMULATIONS_NON_LINEAR_PREDICTIVE')

# This is the path where Code 1 saves the HDF5 files
results_path = base_path  # Updated path to match Code 1

# List of factors used in the simulations
factors = [1, 3, 4, 5, 6]

# Dictionary to store the loaded scores
loaded_scores_dict = {}

# Load the scores for each factor
for n_factors in factors:
    hdf5_path = results_path / f'scores_{n_factors}.h5'
    try:
        # Load the scores from the HDF5 file
        scores_df = pd.read_hdf(str(hdf5_path), key='scores')
        loaded_scores_dict[n_factors] = scores_df
        print(f"Scores successfully loaded for {n_factors} factors:")
        print(scores_df)
    except Exception as e:
        print(f"Error loading scores from {hdf5_path}: {e}")

# Access the loaded scores for further analysis
# Example: Access scores for a specific factor
specific_factor = 3
if specific_factor in loaded_scores_dict:
    specific_scores_df = loaded_scores_dict[specific_factor]
    # Perform operations on specific_scores_df
    print(f"Scores for factor {specific_factor}:")
    print(specific_scores_df)
else:
    print(f"No scores found for factor {specific_factor}")


In [ ]:
from pathlib import Path

base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIMULATIONS_NON_LINEAR_PREDICTIVE')
results_path = base_path / 'results' / 'asset_pricing'

if not results_path.exists():
    print(f"Creating directory: {results_path}")
    results_path.mkdir(parents=True)
else:
    print(f"Directory already exists: {results_path}")

# Check if directory creation was successful
if results_path.exists():
    print(f"Directory successfully created or already exists: {results_path}")
else:
    print(f"Failed to create directory: {results_path}")


#**IPCA LINEAR TOTAL**

In [ ]:
!pip install tables
!pip install ipca

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from ipca import InstrumentedPCA
from pathlib import Path
from tqdm import tqdm
from google.colab import drive
import os

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIMULATIONS_LINEAR_TOTAL')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    print(f"Creating directory: {results_path}")
    results_path.mkdir(parents=True)
else:
    print(f"Directory already exists: {results_path}")

# Check if the directory was created successfully
if results_path.exists():
    print(f"Directory successfully created: {results_path}")
else:
    print(f"Failed to create directory: {results_path}")

# Parameters
N = 200
T = 180
Pc = 50
Px = 50
num_factors = 5
R2_target = 0.45
num_simulations = 100
factors = [1,2,3,4,5,6]

cols = ['Factors', 'Sharpe Ratio x_t', 'Long-Short Portfolio Sharpe Ratio', 'R^2 for r_t', 'Adjusted R^2', 'R^2 for x_t']

# Function to generate data
def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                         c_normalized[:, 1, t],
                                         0.8 * c_normalized[:, 2, t],
                                         0.6 * c_normalized[:, 3, t],
                                         0.4 * c_normalized[:, 4, t]]).T

    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)
    test_size = T - train_size - val_size - 1
    train_data = r_adjusted[:, :train_size]
    val_data = r_adjusted[:, train_size:train_size + val_size]
    test_data = r_adjusted[:, train_size + val_size:]
    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size

# Helper functions
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[2]):
        Z_t = Z_t_minus_1[:, :, t]
        r_t_i = r_t[:, t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

def calculate_sharpe_ratio(returns, risk_free_rate=0):
    mean_return = np.mean(returns)
    std_dev_return = np.std(returns)
    sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
    return sharpe_ratio

def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    N = predicted_returns.shape[0]
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold
    long_returns = actual_returns[long_positions].mean()
    short_returns = actual_returns[short_positions].mean()
    portfolio_return = long_returns - short_returns
    return portfolio_return

# Loop over each factor and calculate average R^2 for all simulations
scores_dict = {}
for n_factors in tqdm(factors):
    r2_scores = []
    scores = []
    for i in tqdm(range(num_simulations)):
        train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)
        X_train = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(1, train_size + 1)], axis=0)
        y_train = np.concatenate([train_data[:, t] for t in range(train_size)], axis=0)
        X_test = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(train_size + val_size + 1, T)], axis=0)
        y_test = np.concatenate([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)], axis=0)
        indices_train = np.array([[i, t] for t in range(1, train_size + 1) for i in range(N)])
        indices_test = np.array([[i, t] for t in range(train_size + val_size + 1, T) for i in range(N)])
        X_train_df = pd.DataFrame(X_train, index=pd.MultiIndex.from_arrays([indices_train[:, 0], indices_train[:, 1]], names=['entity', 'time']))
        y_train_series = pd.Series(y_train, index=pd.MultiIndex.from_arrays([indices_train[:, 0], indices_train[:, 1]], names=['entity', 'time']))
        X_test_df = pd.DataFrame(X_test, index=pd.MultiIndex.from_arrays([indices_test[:, 0], indices_test[:, 1]], names=['entity', 'time']))
        y_test_series = pd.Series(y_test, index=pd.MultiIndex.from_arrays([indices_test[:, 0], indices_test[:, 1]], names=['entity', 'time']))
        regr = InstrumentedPCA(n_factors=n_factors, intercept=False, max_iter=2000, iter_tol=1e-05, alpha=0.0, l1_ratio=1.0, n_jobs=-1, backend='loky')
        regr.fit(X=X_train_df, y=y_train_series)

        try:
            y_pred = regr.predict(X=X_test_df, mean_factor=True)
            y_pred = np.array(y_pred).flatten()
            r2 = r2_score(y_test_series, y_pred)
            r2_scores.append(r2)
            Z_t_minus_1_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_size + 1, T)])
            r_t_test = np.array([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)])
            factor_network_input_test = calculate_factor_network_input(Z_t_minus_1_test, r_t_test)
            sharpe_ratio_x_t = calculate_sharpe_ratio(factor_network_input_test)
            y_pred_reshaped = y_pred.reshape((r_t_test.shape[0], r_t_test.shape[1]))
            factor_network_input_pred = calculate_factor_network_input(Z_t_minus_1_test, y_pred_reshaped)
            r2_x_t = r2_score(factor_network_input_test, factor_network_input_pred)
            portfolio_returns = []
            for t in range(r_t_test.shape[1]):
                actual_returns = r_t_test[:, t]
                predicted_returns = y_pred_reshaped[:, t]
                if predicted_returns.shape == actual_returns.shape:
                    portfolio_return = calculate_long_short_portfolio(predicted_returns, actual_returns)
                    portfolio_returns.append(portfolio_return)
                else:
                    print(f"Shape mismatch: predicted_returns.shape = {predicted_returns.shape}, actual_returns.shape = {actual_returns.shape}")
            portfolio_returns = np.array(portfolio_returns)
            long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)
            prediction_df = pd.DataFrame({
                'predicted returns': y_pred,
                'actual returns': y_test_series
            })
            print(f'| FACTOR : {n_factors} | SIMULATION: {i} | TEST R^2, r_t: {r2} |TEST R^2, x_t : {r2_x_t} ')
            print(f"Simulation {i+1}, Factors {n_factors} - R-squared (R²): {r2}")
            scores.append([n_factors, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, r2, adjusted_R2, r2_x_t])
        except ValueError as e:
            print(f"Simulation {i+1}, Factors {n_factors} - Error during prediction: {e}")

    # Save scores to HDF5
    scores_df = pd.DataFrame(scores, columns=cols)
    scores_dict[n_factors] = scores_df
    hdf5_path = results_path / f'scores_{n_factors}.h5'
    print(f"Saving scores to {hdf5_path}")
    try:
        scores_df.to_hdf(str(hdf5_path), key='scores', mode='w')
        print(f"Scores successfully saved to {hdf5_path}")
    except Exception as e:
        print(f"Error saving scores to {hdf5_path}: {e}")


In [ ]:
# Import necessary libraries
from google.colab import drive
import pandas as pd
from pathlib import Path
!pip install tables

# Mount Google Drive
drive.mount('/content/drive')

# Check if the drive is mounted
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIMULATIONS_LINEAR_TOTAL')
if base_path.exists():
    print("Google Drive mounted successfully and base path exists.")
else:
    print("Failed to mount Google Drive or base path does not exist.")

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# List of factors to load
factors = [1, 2, 3, 4, 5, 6]

# Dictionary to store the loaded scores
loaded_scores = {}

# Load the scores for each factor
for n_factors in factors:
    hdf5_path = results_path / f'scores_{n_factors}.h5'
    if hdf5_path.exists():
        try:
            scores_df = pd.read_hdf(str(hdf5_path), key='scores')
            loaded_scores[n_factors] = scores_df
            print(f"Scores successfully loaded from {hdf5_path}")
        except Exception as e:
            print(f"Error loading scores from {hdf5_path}: {e}")
    else:
        print(f"File does not exist: {hdf5_path}")

# Display the loaded scores
for n_factors, scores_df in loaded_scores.items():
    print(f"Scores for {n_factors} factors:")
    print(scores_df)


In [ ]:
# # Import necessary libraries
# from google.colab import drive
# import pandas as pd
# from pathlib import Path
# !pip install tables
# # Mount Google Drive
# drive.mount('/content/drive')

# # Check if the drive is mounted
# base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIMULATIONS_LINEAR_TOTAL')
# if base_path.exists():
#     print("Google Drive mounted successfully and base path exists.")
# else:
#     print("Failed to mount Google Drive or base path does not exist.")

# # Define the results path relative to the base path
# results_path = base_path / 'results' / 'asset_pricing'

# # List of factors to load
# factors = [1,2, 3, 4, 5, 6]

# # Dictionary to store the loaded scores
# loaded_scores = {}

# # Load the scores for each factor
# for n_factors in factors:
#     hdf5_path = results_path / f'scores_{n_factors}.h5'
#     if hdf5_path.exists():
#         try:
#             scores_df = pd.read_hdf(str(hdf5_path), key='scores')
#             loaded_scores[n_factors] = scores_df
#             print(f"Scores successfully loaded from {hdf5_path}")
#         except Exception as e:
#             print(f"Error loading scores from {hdf5_path}: {e}")
#     else:
#         print(f"File does not exist: {hdf5_path}")

# # Display the loaded scores
# for n_factors, scores_df in loaded_scores.items():
#     print(f"Scores for {n_factors} factors:")
#     print(scores_df)


In [ ]:
# Import necessary libraries
from google.colab import drive
import pandas as pd
from pathlib import Path
!pip install tables

# Mount Google Drive
drive.mount('/content/drive')

# Check if the drive is mounted
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIMULATIONS_LINEAR_TOTAL')
if base_path.exists():
    print("Google Drive mounted successfully and base path exists.")
else:
    print("Failed to mount Google Drive or base path does not exist.")

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# List of factors to load
factors = [1, 2, 3, 4, 5, 6]

# Function to count saved scores
def count_saved_scores(path):
    count = 0
    for file in path.glob('scores_*.h5'):
        count += 1
    return count

# Count the number of saved scores
num_saved_scores = count_saved_scores(results_path)

# Print the number of saved scores
print(f"Number of saved score files: {num_saved_scores}")


# **Non-Linear Model - IPCA Predictive $R^2$**

In [ ]:
!pip install ipca
!pip install tables

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from ipca import InstrumentedPCA
from pathlib import Path
from tqdm import tqdm
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')


##########################################
## DEFINING THE PATH TO SAVE RESULTS IN ##
##########################################

results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIMULATIONS_NON_LINEAR_PREDICTIVE')
hdf5_path = results_path / 'asset_pricing.h5'

if not results_path.exists():
    results_path.mkdir(parents=True)

if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()

##########################################
## DEFINE DATA GENERATING PROCESS       ##
##########################################

def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

    # Normalize characteristics
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

    # Generate factors and errors
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))

    # Define linear factor loadings
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([
            np.power(1.2 * c_normalized[:, 0, t], 3),
            c_normalized[:, 1, t] * c_normalized[:, 0, t],
            np.power(0.4 * c_normalized[:, 2, t], 2),
            2 * c_normalized[:, 3, t] * c_normalized[:, 2, t],
            np.power(0.6 * c_normalized[:, 4, t], 4)
        ]).T

    # Initial Returns Calculation
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]

    # Calculate initial R2
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)

    # Adjust the variances to achieve the target R2
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

    # Generate adjusted returns
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]

    # Calculate adjusted R2
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)

    # Split data into training, validation, and testing sets
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)
    test_size = T - train_size - val_size - 1

    train_data = r_adjusted[:, :train_size]
    val_data = r_adjusted[:, train_size:train_size + val_size]
    test_data = r_adjusted[:, train_size + val_size:]

    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size

######################
## HELPER FUNCTIONS ##
######################

def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[2]):
        Z_t = Z_t_minus_1[:, :, t]
        r_t_i = r_t[:, t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

def calculate_sharpe_ratio(returns, risk_free_rate=0):
    mean_return = np.mean(returns)
    std_dev_return = np.std(returns)
    sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
    return sharpe_ratio

def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    N = predicted_returns.shape[0]
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold
    long_returns = actual_returns[long_positions].mean()
    short_returns = actual_returns[short_positions].mean()
    portfolio_return = long_returns - short_returns
    return portfolio_return

def calculate_forward_returns(returns, forward_period=1):
    return returns[:, forward_period:]

########################
## MAIN SCRIPT        ##
########################

# Seed for reproducibility
np.random.seed(42)

# Constants
N = 200
T = 180
Pc = 50
Px = 50
num_factors = 5
R2_target = 0.45
num_simulations = 100
factors = [1,2,3,4,5,6]

cols = ['Factors', 'Sharpe Ratio x_t', 'Long-Short Portfolio Sharpe Ratio', 'R^2 for r_t', 'Adjusted R^2', 'R^2 for x_t']
scores_dict = {}

for n_factors in tqdm(factors):  # Adjust the range as needed
    r2_scores = []
    scores = []
    for i in tqdm(range(num_simulations)):
        train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

        # Calculate forward returns
        train_forward_returns = calculate_forward_returns(train_data, forward_period=1)
        val_forward_returns = calculate_forward_returns(val_data, forward_period=1)
        test_forward_returns = calculate_forward_returns(test_data, forward_period=1)

        # Adjust the time range to match forward returns
        X_train = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(1, train_size)], axis=0)
        y_train = np.concatenate([train_forward_returns[:, t] for t in range(train_size - 1)], axis=0)
        X_test = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(train_size + val_size + 1, T - 1)], axis=0)
        y_test = np.concatenate([test_forward_returns[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T - 1)], axis=0)

        # Create panel indices
        indices_train = np.array([[i, t] for t in range(1, train_size) for i in range(N)])
        indices_test = np.array([[i, t] for t in range(train_size + val_size + 1, T - 1) for i in range(N)])

        # Convert to pandas DataFrame for better handling (optional)
        X_train_df = pd.DataFrame(X_train, index=pd.MultiIndex.from_arrays([indices_train[:, 0], indices_train[:, 1]], names=['entity', 'time']))
        y_train_series = pd.Series(y_train, index=pd.MultiIndex.from_arrays([indices_train[:, 0], indices_train[:, 1]], names=['entity', 'time']))
        X_test_df = pd.DataFrame(X_test, index=pd.MultiIndex.from_arrays([indices_test[:, 0], indices_test[:, 1]], names=['entity', 'time']))
        y_test_series = pd.Series(y_test, index=pd.MultiIndex.from_arrays([indices_test[:, 0], indices_test[:, 1]], names=['entity', 'time']))

        # Initialize the IPCA model
        regr = InstrumentedPCA(n_factors=n_factors, intercept=False, max_iter=2000, iter_tol=1e-05, alpha=0.0, l1_ratio=1.0, n_jobs=-1, backend='loky')

        # Fit the model on the training data
        regr.fit(X=X_train_df, y=y_train_series)

        try:
            # Generate out-of-sample predictions
            y_pred = regr.predict(X=X_test_df, mean_factor=True)
            y_pred = np.array(y_pred).flatten()

            # Calculate R^2 score
            r2 = r2_score(y_test_series, y_pred)
            print(f'TEST R T R^2 {r2}')

            r2_scores.append(r2)

            # Adjust shapes for factor network input calculation
            Z_t_minus_1_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_size + 1, T - 1)])
            r_t_test = np.array([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T - 1)])

            factor_network_input_test = calculate_factor_network_input(Z_t_minus_1_test, r_t_test)

            sharpe_ratio_x_t = calculate_sharpe_ratio(factor_network_input_test)

            # Reshape y_pred to match r_t_test shape
            y_pred_reshaped = y_pred.reshape((r_t_test.shape[0], r_t_test.shape[1]))

            factor_network_input_pred = calculate_factor_network_input(Z_t_minus_1_test, y_pred_reshaped)

            r2_x_t = r2_score(factor_network_input_test, factor_network_input_pred)
            print(f'TEST X T R^2 {r2_x_t}')
            portfolio_returns = []
            for t in range(r_t_test.shape[1]):
                actual_returns = r_t_test[:, t]
                predicted_returns = y_pred_reshaped[:, t]

                # Ensure the predicted and actual returns have the same shape
                if predicted_returns.shape == actual_returns.shape:
                    portfolio_return = calculate_long_short_portfolio(predicted_returns, actual_returns)
                    portfolio_returns.append(portfolio_return)
                else:
                    print(f"Shape mismatch: predicted_returns.shape = {predicted_returns.shape}, actual_returns.shape = {actual_returns.shape}")

            portfolio_returns = np.array(portfolio_returns)
            long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)

            # Save scores
            scores.append([n_factors, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, r2, adjusted_R2, r2_x_t])

        except ValueError as e:
            print(f"Simulation {i+1}, Factors {n_factors} - Error during prediction: {e}")

    # Calculate and print the average R^2 score for the current number of factors
    average_r2 = np.mean(r2_scores)
    print(f"Average R-squared (R²) for {n_factors} factors across all simulations: {average_r2}")

    # Save scores to HDF5
    scores_df = pd.DataFrame(scores, columns=cols)
    scores_dict[n_factors] = scores_df
    save_path = results_path / f'scores_{n_factors}.h5'
    print(f"Saving scores to {save_path}")  # Debug: Print the save path
    scores_df.to_hdf(str(save_path), key='scores', mode='w')


#############################
## FUNCTION TO LOAD SCORES ##
#############################

def load_scores(results_path, num_factors):
    scores_dict = {}
    for n_factors in range(1, num_factors + 1):
        file_path = results_path / f'scores_{n_factors}.h5'
        print(f"Checking file: {file_path}")  # Debug: Check the file path
        if file_path.exists():
            print(f"Loading scores for {n_factors} factors from {file_path}")  # Debug: File exists
            try:
                scores_df = pd.read_hdf(file_path, key='scores')
                scores_dict[n_factors] = scores_df
                print(f"Loaded scores for {n_factors} factors")  # Debug: Successfully loaded
            except Exception as e:
                print(f"Error loading {file_path}: {e}")  # Debug: Error during loading
        else:
            print(f"File {file_path} does not exist")  # Debug: File does not exist
    return scores_dict

# Example usage to load the scores
scores_dict_loaded = load_scores(results_path, num_factors=6)

# Print the loaded scores
for n_factors, scores_df in scores_dict_loaded.items():
    print(f"\nScores for {n_factors} factors:")
    print(scores_df.head())

**Load Scores**

In [ ]:
from pathlib import Path
from tqdm import tqdm
from google.colab import drive
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from google.colab import drive

!pip install tables

# Mount Google Drive
drive.mount('/content/drive')

factors = [1,2,3,4,5,6]


##########################################
## DEFINING THE PATH TO SAVE RESULTS IN ##
##########################################

results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIMULATIONS_NON_LINEAR_PREDICTIVE')
hdf5_path = results_path / 'asset_pricing.h5'

if not results_path.exists():
    results_path.mkdir(parents=True)

if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()



#############################
## FUNCTION TO LOAD SCORES ##
#############################

def load_scores(results_path, num_factors):
    scores_dict = {}
    for n_factors in range(1, num_factors + 1):
        file_path = results_path / f'scores_{n_factors}.h5'
        print(f"Checking file: {file_path}")  # Debug: Check the file path
        if file_path.exists():
            print(f"Loading scores for {n_factors} factors from {file_path}")  # Debug: File exists
            try:
                scores_df = pd.read_hdf(file_path, key='scores')
                scores_dict[n_factors] = scores_df
                print(f"Loaded scores for {n_factors} factors")  # Debug: Successfully loaded
            except Exception as e:
                print(f"Error loading {file_path}: {e}")  # Debug: Error during loading
        else:
            print(f"File {file_path} does not exist")  # Debug: File does not exist
    return scores_dict

# Example usage to load the scores
scores_dict_loaded = load_scores(results_path, num_factors=6)

# Print the loaded scores
for n_factors, scores_df in scores_dict_loaded.items():
    print(f"\nScores for {n_factors} factors:")
    print(scores_df.head())

# **Linear IPCA PREDICTIVE $R^2$**

In [ ]:
!pip install ipca
!pip install tables
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from ipca import InstrumentedPCA
from pathlib import Path
from tqdm import tqdm
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')


##########################################
## DEFINING THE PATH TO SAVE RESULTS IN ##
##########################################

results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIMULATIONS_LINEAR_PREDICTIVE')
hdf5_path = results_path / 'asset_pricing.h5'

if not results_path.exists():
    results_path.mkdir(parents=True)

if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()

##########################################
## DEFINE DATA GENERATING PROCESS       ##
##########################################

def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

    # Normalize characteristics
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

    # Generate factors and errors
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))

    # Define linear factor loadings
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                         c_normalized[:, 1, t],
                                         0.8 * c_normalized[:, 2, t],
                                         0.6 * c_normalized[:, 3, t],
                                         0.4 * c_normalized[:, 4, t]]).T

    # Initial Returns Calculation
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]

    # Calculate initial R2
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)

    # Adjust the variances to achieve the target R2
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

    # Generate adjusted returns
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]

    # Calculate adjusted R2
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)

    # Split data into training, validation, and testing sets
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)
    test_size = T - train_size - val_size - 1

    train_data = r_adjusted[:, :train_size]
    val_data = r_adjusted[:, train_size:train_size + val_size]
    test_data = r_adjusted[:, train_size + val_size:]

    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size

######################
## HELPER FUNCTIONS ##
######################

def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[2]):
        Z_t = Z_t_minus_1[:, :, t]
        r_t_i = r_t[:, t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

def calculate_sharpe_ratio(returns, risk_free_rate=0):
    mean_return = np.mean(returns)
    std_dev_return = np.std(returns)
    sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
    return sharpe_ratio

def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    N = predicted_returns.shape[0]
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold
    long_returns = actual_returns[long_positions].mean()
    short_returns = actual_returns[short_positions].mean()
    portfolio_return = long_returns - short_returns
    return portfolio_return

def calculate_forward_returns(returns, forward_period=1):
    return returns[:, forward_period:]

########################
## MAIN SCRIPT        ##
########################

# Seed for reproducibility
np.random.seed(42)

# Constants
N = 200
T = 180
Pc = 50
Px = 50
num_factors = 5
R2_target = 0.45
num_simulations = 100

cols = ['Factors', 'Sharpe Ratio x_t', 'Long-Short Portfolio Sharpe Ratio', 'R^2 for r_t', 'Adjusted R^2', 'R^2 for x_t']
scores_dict = {}

# Loop over each factor and calculate average R^2 for all simulations
for n_factors in tqdm(range(1, 7)):  # Adjust the range as needed
    r2_scores = []
    scores = []
    for i in tqdm(range(num_simulations)):
        train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

        # Calculate forward returns
        train_forward_returns = calculate_forward_returns(train_data, forward_period=1)
        val_forward_returns = calculate_forward_returns(val_data, forward_period=1)
        test_forward_returns = calculate_forward_returns(test_data, forward_period=1)

        # Adjust the time range to match forward returns
        X_train = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(1, train_size)], axis=0)
        y_train = np.concatenate([train_forward_returns[:, t] for t in range(train_size - 1)], axis=0)
        X_test = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(train_size + val_size + 1, T - 1)], axis=0)
        y_test = np.concatenate([test_forward_returns[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T - 1)], axis=0)

        # Create panel indices
        indices_train = np.array([[i, t] for t in range(1, train_size) for i in range(N)])
        indices_test = np.array([[i, t] for t in range(train_size + val_size + 1, T - 1) for i in range(N)])

        # Convert to pandas DataFrame for better handling (optional)
        X_train_df = pd.DataFrame(X_train, index=pd.MultiIndex.from_arrays([indices_train[:, 0], indices_train[:, 1]], names=['entity', 'time']))
        y_train_series = pd.Series(y_train, index=pd.MultiIndex.from_arrays([indices_train[:, 0], indices_train[:, 1]], names=['entity', 'time']))
        X_test_df = pd.DataFrame(X_test, index=pd.MultiIndex.from_arrays([indices_test[:, 0], indices_test[:, 1]], names=['entity', 'time']))
        y_test_series = pd.Series(y_test, index=pd.MultiIndex.from_arrays([indices_test[:, 0], indices_test[:, 1]], names=['entity', 'time']))

        # Initialize the IPCA model
        regr = InstrumentedPCA(n_factors=n_factors, intercept=False, max_iter=2000, iter_tol=1e-05, alpha=0.0, l1_ratio=1.0, n_jobs=-1, backend='loky')

        # Fit the model on the training data
        regr.fit(X=X_train_df, y=y_train_series)

        try:
            # Generate out-of-sample predictions
            y_pred = regr.predict(X=X_test_df, mean_factor=True)
            y_pred = np.array(y_pred).flatten()

            # Calculate R^2 score
            r2 = r2_score(y_test_series, y_pred)
            print(f'TEST R T R^2 {r2}')

            r2_scores.append(r2)

            # Adjust shapes for factor network input calculation
            Z_t_minus_1_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_size + 1, T - 1)])
            r_t_test = np.array([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T - 1)])

            factor_network_input_test = calculate_factor_network_input(Z_t_minus_1_test, r_t_test)

            sharpe_ratio_x_t = calculate_sharpe_ratio(factor_network_input_test)

            # Reshape y_pred to match r_t_test shape
            y_pred_reshaped = y_pred.reshape((r_t_test.shape[0], r_t_test.shape[1]))

            factor_network_input_pred = calculate_factor_network_input(Z_t_minus_1_test, y_pred_reshaped)

            r2_x_t = r2_score(factor_network_input_test, factor_network_input_pred)
            print(f'TEST X T R^2 {r2_x_t}')
            portfolio_returns = []
            for t in range(r_t_test.shape[1]):
                actual_returns = r_t_test[:, t]
                predicted_returns = y_pred_reshaped[:, t]

                # Ensure the predicted and actual returns have the same shape
                if predicted_returns.shape == actual_returns.shape:
                    portfolio_return = calculate_long_short_portfolio(predicted_returns, actual_returns)
                    portfolio_returns.append(portfolio_return)
                else:
                    print(f"Shape mismatch: predicted_returns.shape = {predicted_returns.shape}, actual_returns.shape = {actual_returns.shape}")

            portfolio_returns = np.array(portfolio_returns)
            long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)

            # Save scores
            scores.append([n_factors, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, r2, adjusted_R2, r2_x_t])

        except ValueError as e:
            print(f"Simulation {i+1}, Factors {n_factors} - Error during prediction: {e}")

    # Calculate and print the average R^2 score for the current number of factors
    average_r2 = np.mean(r2_scores)
    print(f"Average R-squared (R²) for {n_factors} factors across all simulations: {average_r2}")

    # Save scores to HDF5
    scores_df = pd.DataFrame(scores, columns=cols)
    print(f"Scores DataFrame for {n_factors} factors:\n{scores_df.head()}")  # Debugging print
    scores_dict[n_factors] = scores_df
    scores_df.to_hdf(str(results_path / f'scores_{n_factors}.h5'), key='scores', mode='w')

#############################
## FUNCTION TO LOAD SCORES ##
#############################

def load_scores(results_path, num_factors):
    scores_dict = {}
    for n_factors in range(1, num_factors+1):
        file_path = results_path / f'scores_{n_factors}.h5'
        if file_path.exists():
            scores_df = pd.read_hdf(file_path, key='scores')
            scores_dict[n_factors] = scores_df
    return scores_dict

# Example usage to load the scores
scores_dict_loaded = load_scores(results_path, num_factors=6)
for n_factors, scores_df in scores_dict_loaded.items():
    print(f"Scores for {n_factors} factors:")
    print(scores_df.head())


**Load predictions**

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive
!pip install tables
# Mount Google Drive
drive.mount('/content/drive')

results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIMULATIONS_LINEAR_PREDICTIVE')

def load_scores(results_path, num_factors):
    scores_dict = {}
    for n_factors in range(1, num_factors+1):
        file_path = results_path / f'scores_{n_factors}.h5'
        if file_path.exists():
            print(f"Loading scores from {file_path}")
            scores_df = pd.read_hdf(file_path, key='scores')
            scores_dict[n_factors] = scores_df
        else:
            print(f"File {file_path} does not exist")
    return scores_dict

# Example usage to load the scores
scores_dict_loaded = load_scores(results_path, num_factors=6)
for n_factors, scores_df in scores_dict_loaded.items():
    print(f"Scores for {n_factors} factors:")
    print(scores_df.head())


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from pathlib import Path
from tqdm import tqdm
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

factors = [1,2,3,4,5,6]
##########################################
## DEFINING THE PATH TO SAVE RESULTS IN ##
##########################################

results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIMULATIONS_LINEAR_PREDICTIVE')
hdf5_path = results_path / 'asset_pricing.h5'

if not results_path.exists():
    results_path.mkdir(parents=True)

if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()


#############################
## FUNCTION TO LOAD SCORES ##
#############################

def load_scores(results_path, num_factors):
    scores_dict = {}
    for n_factors in range(1, num_factors+1):
        file_path = results_path / f'scores_{n_factors}.h5'
        if file_path.exists():
            print(f"Loading scores from {file_path}")
            scores_df = pd.read_hdf(file_path, key='scores')
            scores_dict[n_factors] = scores_df
        else:
            print(f"File {file_path} does not exist")
    return scores_dict

# Example usage to load the scores
scores_dict_loaded = load_scores(results_path, num_factors=6)
for n_factors, scores_df in scores_dict_loaded.items():
    print(f"Scores for {n_factors} factors:")
    print(scores_df.head())


In [ ]:
scores_df

In [ ]:
from pathlib import Path

results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIM_PRED')

# List files in the directory
print("Files in the directory:")
for file in results_path.iterdir():
    print(file)


In [ ]:
import pandas as pd
from pathlib import Path

##########################################
## DEFINING THE PATH TO SAVE RESULTS IN ##
##########################################

results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIM_PRED')

#############################
## FUNCTION TO LOAD SCORES ##
#############################

def load_scores(results_path, num_factors):
    scores_dict = {}
    cols = ['Factors', 'Sharpe Ratio x_t', 'Long-Short Portfolio Sharpe Ratio', 'R^2 for r_t', 'Adjusted R^2', 'R^2 for x_t']
    for n_factors in range(1, num_factors + 1):
        file_path = results_path / f'scores_{n_factors}.h5'
        if file_path.exists():
            scores_df = pd.read_hdf(file_path, key='scores')
            scores_df.columns = cols  # Ensure the correct column names are used
            scores_dict[n_factors] = scores_df
    return scores_dict

# Example usage to load the scores
scores_dict_loaded = load_scores(results_path, num_factors=6)
for n_factors, scores_df in scores_dict_loaded.items():
    print(f"Scores for {n_factors} factors:")
    print(scores_df.head())


In [ ]:
scores_dict_loaded

In [ ]:
print(y_test_series.shape, y_pred.shape)

print(r2)

In [ ]:
y_val_pred.shape

In [ ]:
y_train_series.shape

In [ ]:
X_train_df

In [ ]:
print(X_train.shape,X_val.shape)
print(y_train.shape,y_val.shape)

print(X_train_df.shape,X_val_df.shape)
print(y_train_series.shape,y_val_series.shape)

In [ ]:
print(len(indices_val))
print(len(indices_train))
print(len(indices_test))

In [ ]:
# Access scores for a specific number of factors from the dictionary
n_factors = 1
if n_factors in scores_dict:
    scores_df = scores_dict[n_factors]
    print(scores_df)
else:
    print(f"No scores found for {n_factors} factors.")


In [ ]:

print(X_train_df.shape)
print(y_train_series.shape)

print(type(X_train_df))
print(type(y_train_series.shape))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from ipca import InstrumentedPCA
from pathlib import Path
from tqdm import tqdm

# Seed for reproducibility
np.random.seed(42)

# Constants
N = 200
T = 180
Pc = 50
Px = 50
num_factors = 5
R2_target = 0.45
num_simulations = 100

# Define functions
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[0]):
        Z_t = Z_t_minus_1[t]
        r_t_i = r_t[t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

def calculate_sharpe_ratio(returns, risk_free_rate=0):
    mean_return = np.mean(returns)
    std_dev_return = np.std(returns)
    sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
    return sharpe_ratio

def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    N = predicted_returns.shape[0]
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold
    long_returns = actual_returns[long_positions].mean(axis=0)
    short_returns = actual_returns[short_positions].mean(axis=0)
    portfolio_return = long_returns - short_returns
    return portfolio_return

# Path to save the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_IPCA')
hdf5_path = results_path / 'fwd_predictions.h5'

if not results_path.exists():
    results_path.mkdir(parents=True)

if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()

cols = ['Factors', 'Sharpe Ratio x_t', 'Long-Short Portfolio Sharpe Ratio', 'R^2 for r_t', 'Adjusted R^2', 'R^2 for x_t']
scores_dict = {}

# Loop over each factor and calculate average R^2 for all simulations
for n_factors in tqdm(range(1, 7)):  # Adjust the range as needed
    r2_scores = []
    scores = []
    for i in tqdm(range(num_simulations)):
        train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

        X_train = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(1, train_size + 1)], axis=0)
        y_train = np.concatenate([train_data[:, t] for t in range(train_size)], axis=0)
        X_test = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(train_size + val_size + 1, T)], axis=0)
        y_test = np.concatenate([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)], axis=0)

        indices_train = np.array([[i, t] for t in range(1, train_size + 1) for i in range(N)])
        indices_test = np.array([[i, t] for t in range(train_size + val_size + 1, T) for i in range(N)])

        X_train_df = pd.DataFrame(X_train, index=pd.MultiIndex.from_arrays([indices_train[:, 0], indices_train[:, 1]], names=['entity', 'time']))
        y_train_series = pd.Series(y_train, index=pd.MultiIndex.from_arrays([indices_train[:, 0], indices_train[:, 1]], names=['entity', 'time']))
        X_test_df = pd.DataFrame(X_test, index=pd.MultiIndex.from_arrays([indices_test[:, 0], indices_test[:, 1]], names=['entity', 'time']))
        y_test_series = pd.Series(y_test, index=pd.MultiIndex.from_arrays([indices_test[:, 0], indices_test[:, 1]], names=['entity', 'time']))

        regr = InstrumentedPCA(n_factors=n_factors, intercept=False, max_iter=2000, iter_tol=1e-05, alpha=0.0, l1_ratio=1.0, n_jobs=-1, backend='loky')
        regr.fit(X=X_train_df, y=y_train_series)

        try:
            y_pred = regr.predict(X=X_test_df, mean_factor=True)
            y_pred = np.array(y_pred).flatten()

            r2 = r2_score(y_test_series, y_pred)
            r2_scores.append(r2)

            Z_t_minus_1_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_size + 1, T)])
            r_t_test = np.array([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)])

            factor_network_input = calculate_factor_network_input(Z_t_minus_1_test, r_t_test)

            sharpe_ratio_x_t = calculate_sharpe_ratio(factor_network_input)
            portfolio_returns = np.zeros(r_t_test.shape[1])
            for t in range(r_t_test.shape[1]):
                portfolio_returns[t] = calculate_long_short_portfolio(y_pred, r_t_test)

            long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)

            prediction_df = pd.DataFrame({
                'predicted returns': y_pred,
                'actual returns': y_test_series.values
            })

            print(f'| FACTOR : {n_factors} | SIMULATION: {i} | TEST R^2: {r2} |')
            with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
                store.put(f'fwd_predictions/n_factors_{n_factors}_simulation_{i}', prediction_df, format='table')

            print(f"Simulation {i+1}, Factors {n_factors} - R-squared (R²): {r2}")

            scores.append([n_factors, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, r2, adjusted_R2, r2])

        except ValueError as e:
            print(f"Simulation {i+1}, Factors {n_factors} - Error during prediction: {e}")

    average_r2 = np.mean(r2_scores)
    print(f"Average R-squared (R²) for {n_factors} factors across all simulations: {average_r2}")

    scores_df = pd.DataFrame(scores, columns=cols)
    scores_dict[n_factors] = scores_df

    scores_df.to_hdf(str(results_path / f'scores_{n_factors}.h5'), key='scores', mode='w')


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from ipca import InstrumentedPCA
from pathlib import Path
from tqdm import tqdm

# Seed for reproducibility
np.random.seed(42)

# Constants
N = 200
T = 180
Pc = 50
Px = 50
num_factors = 5
R2_target = 0.45
num_simulations = 100

# Define functions
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[0]):
        Z_t = Z_t_minus_1[t]
        r_t_i = r_t[t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

def calculate_sharpe_ratio(returns, risk_free_rate=0):
    mean_return = np.mean(returns)
    std_dev_return = np.std(returns)
    sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
    return sharpe_ratio

def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    N = predicted_returns.shape[0]
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold
    long_returns = actual_returns[long_positions].mean()
    short_returns = actual_returns[short_positions].mean()
    portfolio_return = long_returns - short_returns
    return portfolio_return

# Path to save the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_IPCA')
hdf5_path = results_path / 'fwd_predictions.h5'

if not results_path.exists():
    results_path.mkdir(parents=True)

if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()

cols = ['Factors', 'Sharpe Ratio x_t', 'Long-Short Portfolio Sharpe Ratio', 'R^2 for r_t', 'Adjusted R^2', 'R^2 for x_t']
scores_dict = {}

# Loop over each factor and calculate average R^2 for all simulations
for n_factors in tqdm(range(1, 7)):  # Adjust the range as needed
    r2_scores = []
    scores = []
    for i in tqdm(range(num_simulations)):
        train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

        X_train = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(1, train_size + 1)], axis=0)
        y_train = np.concatenate([train_data[:, t] for t in range(train_size)], axis=0)
        X_test = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(train_size + val_size + 1, T)], axis=0)
        y_test = np.concatenate([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)], axis=0)

        indices_train = np.array([[i, t] for t in range(1, train_size + 1) for i in range(N)])
        indices_test = np.array([[i, t] for t in range(train_size + val_size + 1, T) for i in range(N)])

        X_train_df = pd.DataFrame(X_train, index=pd.MultiIndex.from_arrays([indices_train[:, 0], indices_train[:, 1]], names=['entity', 'time']))
        y_train_series = pd.Series(y_train, index=pd.MultiIndex.from_arrays([indices_train[:, 0], indices_train[:, 1]], names=['entity', 'time']))
        X_test_df = pd.DataFrame(X_test, index=pd.MultiIndex.from_arrays([indices_test[:, 0], indices_test[:, 1]], names=['entity', 'time']))
        y_test_series = pd.Series(y_test, index=pd.MultiIndex.from_arrays([indices_test[:, 0], indices_test[:, 1]], names=['entity', 'time']))

        regr = InstrumentedPCA(n_factors=n_factors, intercept=False, max_iter=2000, iter_tol=1e-05, alpha=0.0, l1_ratio=1.0, n_jobs=-1, backend='loky')
        regr.fit(X=X_train_df, y=y_train_series)

        try:
            y_pred = regr.predict(X=X_test_df, mean_factor=True)
            y_pred = np.array(y_pred).flatten()

            r2 = r2_score(y_test_series, y_pred)
            r2_scores.append(r2)

            Z_t_minus_1_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_size + 1, T)])
            r_t_test = np.array([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)])

            factor_network_input = calculate_factor_network_input(Z_t_minus_1_test, r_t_test)

            sharpe_ratio_x_t = calculate_sharpe_ratio(factor_network_input)

            # Reshape y_pred to match r_t_test shape
            y_pred_reshaped = y_pred.reshape(N, -1)

            portfolio_returns = []
            for t in range(r_t_test.shape[1]):
                actual_returns = r_t_test[:, t]
                predicted_returns = y_pred_reshaped[:, t]
                portfolio_return = calculate_long_short_portfolio(predicted_returns, actual_returns)
                portfolio_returns.append(portfolio_return)

            portfolio_returns = np.array(portfolio_returns)
            long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)

            prediction_df = pd.DataFrame({
                'predicted returns': y_pred,
                'actual returns': y_test_series.values
            })

            print(f'| FACTOR : {n_factors} | SIMULATION: {i} | TEST R^2: {r2} |')
            with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
                store.put(f'fwd_predictions/n_factors_{n_factors}_simulation_{i}', prediction_df, format='table')

            print(f"Simulation {i+1}, Factors {n_factors} - R-squared (R²): {r2}")

            scores.append([n_factors, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, r2, adjusted_R2, r2])

        except ValueError as e:
            print(f"Simulation {i+1}, Factors {n_factors} - Error during prediction: {e}")

    average_r2 = np.mean(r2_scores)
    print(f"Average R-squared (R²) for {n_factors} factors across all simulations: {average_r2}")

    scores_df = pd.DataFrame(scores, columns=cols)
    scores_dict[n_factors] = scores_df

    scores_df.to_hdf(str(results_path / f'scores_{n_factors}.h5'), key='scores', mode='w')


In [ ]:
print(predicted_returns.shape)
print(actual_returns.shape)


In [ ]:
import numpy as np
import pandas as pd
from ipca import InstrumentedPCA
from sklearn.metrics import r2_score

# Example synthetic data generation (you already have this part)
train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

# Combine train and validation data for training
X_train = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(1, train_size + 1)], axis=0)
y_train = np.concatenate([train_data[:, t] for t in range(train_size)], axis=0)
X_test = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(train_size + val_size + 1, T)], axis=0)
y_test = np.concatenate([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)], axis=0)

# Create panel indices
indices_train = np.array([[i, t] for t in range(1, train_size + 1) for i in range(N)])
indices_test = np.array([[i, t] for t in range(train_size + val_size + 1, T) for i in range(N)])

# Convert to pandas DataFrame for better handling (optional)
X_train_df = pd.DataFrame(X_train, index=pd.MultiIndex.from_arrays([indices_train[:, 0], indices_train[:, 1]], names=['entity', 'time']))
y_train_series = pd.Series(y_train, index=pd.MultiIndex.from_arrays([indices_train[:, 0], indices_train[:, 1]], names=['entity', 'time']))
X_test_df = pd.DataFrame(X_test, index=pd.MultiIndex.from_arrays([indices_test[:, 0], indices_test[:, 1]], names=['entity', 'time']))
y_test_series = pd.Series(y_test, index=pd.MultiIndex.from_arrays([indices_test[:, 0], indices_test[:, 1]], names=['entity', 'time']))

# Initialize the IPCA model
regr = InstrumentedPCA(n_factors=n_factors, intercept=False, max_iter=2000, iter_tol=1e-05, alpha=0.0, l1_ratio=1.0, n_jobs=-1, backend='loky')

# Fit the model on the training data
regr.fit(X=X_train_df, y=y_train_series)

# Generate out-of-sample predictions
y_pred = regr.predict(X=X_test_df, mean_factor=True)

# Calculate R^2 score
r2 = r2_score(y_test_series, y_pred)

print(f"R^2 Score: {r2}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install pandas pyarrow
import pandas as pd

# Define the file path
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# Load the Parquet file into a DataFrame
df = pd.read_parquet(file_path, engine='pyarrow')

# Check the loaded data
print(df.head())


In [ ]:
import pandas as pd

# Reset the index to convert 'month' to a column
df_reset = df.reset_index()

# Ensure the 'month' column is in datetime format
df_reset['month'] = pd.to_datetime(df_reset['month'])

# Set the index back to 'month' and 'permno'
df = df_reset.set_index(['month', 'permno'])

# Sort the MultiIndex to ensure it is lexsorted
df = df.sort_index()

# Define the date range
start_date = '1960-01-01'
end_date = '2020-01-01'

# Filter the DataFrame for the specified date range
df = df.loc[start_date:end_date]

# Display the filtered DataFrame
print(df.index)


In [ ]:
import pandas as pd
from sklearn.preprocessing import quantile_transform
from tqdm import tqdm

# Assuming 'df' is your dataframe that has been filtered and processed so far.

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]

# Function to carry forward annual and quarterly data within a MultiIndex
def carry_forward(data, frequency, index_name):
    if frequency == 'annual':
        period = 12
    elif frequency == 'quarterly':
        period = 3
    else:
        raise ValueError("Frequency should be 'annual' or 'quarterly'")

    carried_data = data.copy()
    for i in range(1, period):
        carried_data = carried_data.groupby(level=index_name).ffill()
    return carried_data

# Apply carry forward for annual and quarterly data with progress bars
for code in tqdm(annual_codes, desc="Processing annual codes"):
    char_column = f'characteristic_{code}'
    df[char_column] = carry_forward(df[char_column], 'annual', 'permno')

for code in tqdm(quarterly_codes, desc="Processing quarterly codes"):
    char_column = f'characteristic_{code}'
    df[char_column] = carry_forward(df[char_column], 'quarterly', 'permno')

# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
narratives = [f'characteristic_{code}' for code in all_codes]

# Enable the tqdm pandas extension
tqdm.pandas()

# Rank normalization function
def rank_normalize(group, columns):
    for col in columns:
        values = group[col].values.reshape(-1, 1)
        if len(values) > 1:  # Ensure there is more than one value to normalize
            transformed_values = quantile_transform(values, copy=True, n_quantiles=min(len(values), 100)).flatten()
            group[col] = transformed_values * 2 - 1
        else:
            group[col] = 0  # Handle case with a single value to avoid all -1.0
    return group

# Reset index to avoid duplicate levels after groupby and progress_apply
df_reset = df.reset_index()

# Apply rank normalization by month with progress bar
df_reset = df_reset.groupby('month').progress_apply(lambda group: rank_normalize(group, columns=narratives))

# Reset index to flatten the MultiIndex generated by groupby-apply
df_reset.reset_index(drop=True, inplace=True)

# Set the correct index
df1 = df_reset.set_index(['month', 'permno'])

# Display the dataframe head to check the result
print(df1.head())


In [ ]:
import pandas as pd

# Ensure the dataframe is sorted by 'permno' and 'month'
df1 = df1.sort_index()

# Shift the 'ret_excess' column to calculate forward returns for the next month
df1['forward_ret_excess_1m'] = df1.groupby('permno')['ret_excess'].shift(-1)

# # If you want to calculate forward returns for the next 3 months
# df1['forward_ret_excess_3m'] = df1.groupby('permno')['ret_excess'].shift(-3)

# # If you want to calculate forward returns for the next 6 months
# df1['forward_ret_excess_6m'] = df1.groupby('permno')['ret_excess'].shift(-6)

# # Display the dataframe head to check the result
# print(df1.head())

# # Optionally, drop rows where forward returns are NaN (at the end of each permno series)
# df1.dropna(subset=['forward_ret_excess_1m', 'forward_ret_excess_3m', 'forward_ret_excess_6m'], how='all', inplace=True)

# Display the dataframe head after dropping NaNs
print(df1.head())


In [ ]:
# import pandas as pd

# # Assuming df1 is your DataFrame

# # Step 1: Calculate the total market capitalization for each month
# df1['total_mktcap'] = df1.groupby('month')['mktcap_lag'].transform('sum')

# # Step 2: Calculate the weight of each stock's market cap relative to the total market cap for that month
# df1['weight'] = df1['mktcap_lag'] / df1['total_mktcap']

# # Step 3: Calculate the market-weighted portfolio return for each stock
# # This involves multiplying the excess return by the weight
# df1['market_weighted_return'] = df1['ret_excess'] * df1['weight']

# # Step 4: Aggregate to get the market-weighted portfolio return for each month
# # Summing these weighted returns will give the market-weighted portfolio return for the month
# df1['market_weighted_portfolio'] = df1.groupby('month')['market_weighted_return'].transform('sum')

# # Display the updated DataFrame to check the results
# print(df1.head())


In [ ]:
import pandas as pd

# Assuming df1 is your DataFrame with a MultiIndex of ['month', 'permno']

# Drop rows with any missing values
df1_clean = df1.dropna()

# Reset index to make 'month' and 'permno' columns for easier grouping
df1_clean_reset = df1_clean.reset_index()

# Group by 'month' and count unique 'permno' entries for each month
permno_counts = df1_clean_reset.groupby('month')['permno'].nunique()

# Calculate the average number of 'permno' entries across all months
average_permno_count = permno_counts.mean()

# Display the average number of 'permno' entries per month
print(f"Average number of permnos per month: {average_permno_count:.2f}")

# Optionally, display the counts for each month
print("Number of permnos per month:")
print(permno_counts.mean)


In [ ]:
# Optionally, display the counts for each month
print("Number of permnos per month:")
print(permno_counts.mean())

In [ ]:
# Rename the index level from 'month' to 'date'
df1_clean.index.rename(['date', 'permno'], inplace=True)


In [ ]:
# Rename the column 'market_weighted_portfolio' to 'characteristic_market_weighted_portfolio'
df1_clean.rename(columns={'market_weighted_portfolio': 'characteristic_market_weighted_portfolio'}, inplace=True)


In [ ]:
# Swap the index levels
df1_clean = df1_clean.swaplevel('date', 'permno').sort_index()



In [ ]:
x_t_all_months = df1_clean

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
#,
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristic_columns = [f'characteristic_{code}' for code in all_codes]

# Add "ret_excess" to the list of columns to select
columns_to_select = characteristic_columns + ['forward_ret_excess_1m']

# Select only the specified columns from merged_df_padded
x_t_all_months = df1_clean[columns_to_select]

n_characeristics = len(characteristic_columns)
n_tickers = x_t_all_months.index.get_level_values('permno').unique()
print(n_characeristics)
print(n_tickers)

# Print the selected DataFrame to verify
print(x_t_all_months.head())


In [ ]:
!pip install ipca

# **PCA**

In [ ]:
import pandas as pd
import numpy as np
from utils import format_time, MultipleTimeSeriesCV
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
import time
from pathlib import Path

# Seed for reproducibility
np.random.seed(42)

# Constants
YEAR = 12

# Initialize the time series cross-validator
cv_portfolio = MultipleTimeSeriesCV(n_splits=20, train_period_length=22 * YEAR, test_period_length=1 * YEAR, lookahead=0 * YEAR)

# Split the data
cv_dataset_splits = list(cv_portfolio.split(x_t_all_months))

# Function to preprocess the train and test data
def get_train_test_data(x_t_all_months, train_idx, test_idx):
    train_set = x_t_all_months.iloc[train_idx]
    test_set = x_t_all_months.iloc[test_idx]

    y_train = train_set['ret_excess'].to_numpy()
    x_train = train_set.drop(columns=['ret_excess'])

    y_test = test_set['ret_excess'].to_numpy()
    x_test = test_set.drop(columns=['ret_excess'])

    # Remove rows with all zero columns in x_train and x_test
    train_all_zeros = (x_train == 0).all(axis=1)
    x_train_filtered = x_train.loc[~train_all_zeros]
    y_train_filtered = y_train[~train_all_zeros]

    test_all_zeros = (x_test == 0).all(axis=1)
    x_test_filtered = x_test.loc[~test_all_zeros]
    y_test_filtered = y_test[~test_all_zeros]

    return x_train_filtered, y_train_filtered, x_test_filtered, y_test_filtered

# Path to save the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_PCA')
hdf5_path = results_path / 'predictions.h5'

# Ensure the results directory exists
if not results_path.exists():
    results_path.mkdir(parents=True)

# Create the HDF5 file initially in 'table' format if it doesn't exist
if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()

# Loop over each factor and calculate average R^2 for all folds
for n_factors in range(1, 7):  # Adjust the range as needed
    r2_scores = []
    for fold, (train_idx, test_idx) in enumerate(cv_dataset_splits):
        x_train, y_train, x_test, y_test = get_train_test_data(x_t_all_months, train_idx, test_idx)

        # Initialize the PCA model
        pca = PCA(n_components=n_factors)

        # Fit the PCA model on the training data to get the latent factors
        x_train_pca = pca.fit_transform(x_train)

        # Fit a regression model on the PCA-transformed data (latent factors)
        from sklearn.linear_model import LinearRegression
        regr = LinearRegression()
        regr.fit(x_train_pca, y_train)

        # Transform the test data using the same PCA
        x_test_pca = pca.transform(x_test)

        # Generate out-of-sample predictions
        y_pred = regr.predict(x_test_pca)

        # Ensure y_pred is a numpy array for consistency
        y_pred = np.array(y_pred).flatten()

        # Compute R^2 score
        r2 = r2_score(y_test, y_pred)
        r2_scores.append(r2)

        # Save the actual and predicted returns
        prediction_df = pd.DataFrame({
            'date': x_test.index.get_level_values('date'),
            'permno': x_test.index.get_level_values('permno').astype(str),
            'predicted returns': y_pred,
            'actual returns': y_test
        }).reset_index(drop=True)

        with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
            store.put(f'predictions/n_factors_{n_factors}_fold_{fold}', prediction_df, format='table')

        print(f"Fold {fold+1}, Factors {n_factors} - R-squared (R²): {r2}")

    # Calculate and print the average R^2 score for the current number of factors
    average_r2 = np.mean(r2_scores)
    print(f"Average R-squared (R²) for {n_factors} factors across all folds: {average_r2}")


**FORWARD RETURNS IPCA**